# 01. Exploratory Data Analysis (EDA) & Problem Formulation

---

## 📌 1. Business Problem Statement

### Context:
Customer attrition (churn) poses a critical financial challenge for retail banks. Acquiring a new customer costs **5 to 25 times more** than retaining an existing one. Furthermore, a **5% increase in customer retention** can boost banking profits by 25% to 95%.

### Objective:
The primary objective of this project is to build an end-to-end Machine Learning pipeline that predicts whether a bank customer is likely to close their account (`Exited = 1`) or stay with the bank (`Exited = 0`).

### Business Impact:
- **Proactive Risk Identification**: Enable the bank's marketing and customer success teams to identify high-risk customers before they churn.
- **Targeted Retention Strategies**: Offer personalized incentives (e.g., promotional interest rates, lowered fee structures, premium service tiers) specifically to churn-prone accounts.
- **Resource Optimization**: Avoid wasting retention budgets on loyal customers who had no intention of leaving.

## ⚙️ 2. Machine Learning Problem Framing

- **Type of Problem**: Binary Classification (Supervised Learning).
- **Target Variable**: `Exited`
  - `0`: Customer stays with the bank (Negative Class / Retained).
  - `1`: Customer leaves the bank (Positive Class / Churned).
- **Primary Metric**: **ROC-AUC & Recall (Sensitivity)**
  - *Why Recall over Accuracy?* In churn prediction, a **False Negative** (failing to identify a customer who is about to churn) is far more costly than a **False Positive** (reaching out to a customer who wasn't going to leave).

## 📊 3. Dataset Overview & Data Dictionary

| Column Name | Type | Description |
| :--- | :--- | :--- |
| `RowNumber` | Integer | Row index (Non-informative identifier) |
| `CustomerId` | Integer | Unique identifier for customer (Non-informative) |
| `Surname` | String | Customer last name (Non-informative) |
| `CreditScore` | Integer | Customer credit score rating (300 - 850) |
| `Geography` | String | Customer location country (France, Spain, Germany) |
| `Gender` | String | Customer gender (Male / Female) |
| `Age` | Integer | Customer age in years |
| `Tenure` | Integer | Number of years the customer has been with the bank |
| `Balance` | Float | Account balance amount |
| `NumOfProducts` | Integer | Number of bank products used (accounts, credit cards, loans) |
| `HasCrCard` | Binary | Does customer own a credit card? (1 = Yes, 0 = No) |
| `IsActiveMember` | Binary | Is customer actively transacting? (1 = Yes, 0 = No) |
| `EstimatedSalary` | Float | Estimated annual salary |
| `Exited` | Binary | **Target Variable**: Churned (1 = Yes, 0 = No) |

---

### Step 1: Import Libraries and Load Raw Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetic plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load dataset from raw folder
raw_data_path = '../data/raw/Churn_Modelling.csv'
df = pd.read_csv(raw_data_path)

print(f"Dataset successfully loaded with shape: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

💡 **Step 1 Insight:**
- The raw dataset consists of **10,000 rows** and **14 columns**.
- The sample preview (`df.head()`) shows metadata features (`RowNumber`, `CustomerId`, `Surname`) alongside demographic, financial, and product features.
- **Actionable Decision:** `RowNumber`, `CustomerId`, and `Surname` are unique per customer and carry no statistical relationship with churn; they will be dropped during feature pipeline construction.

### Step 2: Data Integrity & Health Check

In [ ]:
# Check information schema
df.info()

In [ ]:
# Missing value check & duplicates check
missing_values = df.isnull().sum()
duplicate_count = df.duplicated().sum()

print(f"Total Duplicate Rows: {duplicate_count}")
print("\nMissing Values per Column:")
print(missing_values)

💡 **Step 2 Insight:**
- **Data Completeness:** 0 missing values across all 14 columns and 0 duplicate rows.
- **Data Types:**
  - 11 numerical features (integers and floats).
  - 2 object/string features: `Geography` (France, Spain, Germany) and `Gender` (Female, Male).
- **Actionable Decision:** No missing-value imputation is required, but categorical encoding (One-Hot Encoding) is necessary for `Geography` and `Gender`.

### Step 3: Target Variable Imbalance Check (`Exited`)

In [ ]:
target_counts = df['Exited'].value_counts()
target_pct = df['Exited'].value_counts(normalize=True) * 100

print(f"Retained (0): {target_counts[0]} ({target_pct[0]:.2f}%)")
print(f"Churned  (1): {target_counts[1]} ({target_pct[1]:.2f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=target_pct.index, y=target_pct.values, palette=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Customer Churn Class Distribution (%)', fontsize=14, fontweight='bold')
ax.set_xticklabels(['Retained (0)', 'Churned (1)'])
ax.set_ylabel('Percentage')
plt.show()

💡 **Step 3 Insight (Bar Plot Analysis):**
- **Class Ratio:** **79.63% (7,963 customers)** are Retained (`0`, green bar) vs **20.37% (2,037 customers)** Churned (`1`, red bar).
- **Key Observation:** The 4:1 class imbalance means a naive model predicting 'Retained' for all customers would achieve 79.63% accuracy while missing 100% of churners.
- **Actionable Decision:** Use **Stratified K-Fold CV** to maintain class proportions during split, tune threshold parameters, and optimize for **Recall** and **PR-AUC/ROC-AUC**.

### Step 4: Categorical Feature Analysis (`Geography` & `Gender` vs `Exited`)

In [ ]:
# 1. Absolute Counts Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Geography vs Churn Counts
sns.countplot(data=df, x='Geography', hue='Exited', palette=['#2ecc71', '#e74c3c'], ax=axes[0])
axes[0].set_title('Churn Count by Geography', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Customer Count')

# Gender vs Churn Counts
sns.countplot(data=df, x='Gender', hue='Exited', palette=['#2ecc71', '#e74c3c'], ax=axes[1])
axes[1].set_title('Churn Count by Gender', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Customer Count')

plt.tight_layout()
plt.show()

# 2. Percentage Breakdown Plot (Percentage Churned per Category)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Geography Churn Percentage
geo_pct = df.groupby('Geography')['Exited'].mean().reset_index()
geo_pct['Churn_Pct'] = geo_pct['Exited'] * 100
sns.barplot(data=geo_pct, x='Geography', y='Churn_Pct', palette='Reds_d', ax=axes[0])
axes[0].set_title('Churn Rate (%) by Geography', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 40)
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 8), textcoords='offset points', fontweight='bold')

# Gender Churn Percentage
gender_pct = df.groupby('Gender')['Exited'].mean().reset_index()
gender_pct['Churn_Pct'] = gender_pct['Exited'] * 100
sns.barplot(data=gender_pct, x='Gender', y='Churn_Pct', palette='Reds_d', ax=axes[1])
axes[1].set_title('Churn Rate (%) by Gender', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_ylim(0, 35)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 8), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

💡 **Step 4 Insight (Counts & Percentage Figures Analysis):**
- **Geography Percentage Chart:**
  - **Germany:** High-risk outlier at **32.44% churn rate** (annotated directly above the bar), double that of **France (16.15%)** and **Spain (16.67%)**.
- **Gender Percentage Chart:**
  - **Female:** **25.07% churn rate**, noticeably higher than **Male (16.46% churn rate)**.
- **Actionable Decision:** Both `Geography` and `Gender` display strong non-random separation in churn rates and will be key features for one-hot encoding.

### Step 5: Numerical Feature Distributions & Outlier Analysis

In [ ]:
num_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_features):
    sns.boxplot(data=df, x='Exited', y=col, palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f'{col} distribution by Churn', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

💡 **Step 5 Insight (Boxplots Analysis):**
- **Age Boxplot:** Clear separation between boxes! Retained customers median age is **~36 years** (IQR 31–41), whereas churned customers median age is shifted up to **~45 years** (IQR 38–51). Outliers exist above 60 years in the retained group.
- **NumOfProducts Boxplot:** Extreme contrast! Customers with **2 products** show near-zero churn box width, whereas customers with **3 or 4 products** almost exclusively sit in the churned (`Exited = 1`) category (**82.7%** and **100%** churn rate).
- **Balance Boxplot:** Retained group has a large concentration of zero balances (bottom line of boxplot), whereas churned customers have higher median account balances (~$109,349).
- **CreditScore Boxplot:** Similar distributions between retained and churned groups, though churned customers show extreme low outliers below 400.
- **Tenure & EstimatedSalary Boxplots:** Nearly identical box medians and IQRs across both classes, indicating low individual linear correlation with churn.